In [1]:
import os
import streamlit as st
import sqlite3 
import operator
from dotenv import load_dotenv
from typing import Annotated, List, TypedDict
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode
from langchain.tools import tool

c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
DB_FILE = "crm_database.db" 

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY", "")
openai_api_key = os.getenv("OPENAI_API_KEY", "")

# Checking if both keys have been provided; if not, displays an alert and interrupts execution.
if not groq_api_key or not openai_api_key:
    print("Set both the Groq and OpenAI keys in the sidebar to continue.")

In [3]:
class AgentState(TypedDict):
    # Declaring the 'messages' field as a list of BaseMessage, aggregated by the sum operator.
    messages: Annotated[List[BaseMessage], operator.add]

In [4]:
@tool
def query_crm_database(sql_query: str) -> str:
    # Docstring that describes the behavior and use of the tool
    """
    Executes a SELECT query ONLY on the SQLite CRM database and returns the results.
    We use this tool to obtain information about customers or interactions.
    Available tables:
        1. tb_clients (columns: customer_id, name, email, phone, company, status, created_at)

        - status can be 'Lead', 'Active', 'Inactive', 'Prospect'

        2. tb_interactions (columns: interaction_id, customer_id, interaction_date, type, notes)

        - type can be 'Email', 'Call', 'Meeting', 'Note'

        Important: Provide ONLY `SELECT` SQL queries. Do not use `UPDATE`, `DELETE`, `INSERT`, or `DROP`.

        Example of a valid SQL query:

        'SELECT name, email FROM tb_clients WHERE status = 'Active';' 
        'SELECT i.interaction_date, i.type, i.notes FROM tb_interactions i JOIN tb_clients c ON i.customer_id = c.customer_id WHERE c.name = \\'João Silva\\' ORDER BY i.interaction_date DESC;'
    """

    print(f"--- Tool query_crm_database receiving SQL: {sql_query} ---")

    if not sql_query.strip().upper().startswith("SELECT"):
        print("!!! SECURITY ERROR: Attempting to execute non-SELECT SQL !!!")
        return "Error: This tool can only execute SELECT queries."

    conn = None
    try:

        if not os.path.exists(DB_FILE):
             return f"Error: Database file '{DB_FILE}' not found. Run the script 'create_crm_db.py' first."

        conn = sqlite3.connect(DB_FILE)
        cursor = conn.cursor()
        cursor.execute(sql_query)
        results = cursor.fetchall()

        if not results:
            return "No results were found for the query."
        else:
            column_names = [description[0] for description in cursor.description]
            header = " | ".join(column_names)
            rows_str = [" | ".join(map(str, row)) for row in results]
            max_results = 15
            output = f"Results of the query ({len(results)} found)):\n{header}\n" + "\n".join(rows_str[:max_results])

            if len(results) > max_results:
                output += f"\n... (more {len(results) - max_results} results omitted)"

            return output

    except sqlite3.Error as e:
        print(f"!!! SQL Error: {e} while executing '{sql_query}' !!!")
        return f"Erro ao executar a consulta SQL: {e}. Verifique a sintaxe da sua consulta e os nomes das tabelas/colunas."
   
    except Exception as e:
        print(f"!!! Unexpected ERROR in the tool: {e} !!!")
        return f"An unexpected error occurred in the database tool: {e}"
    
    finally:
        if conn:
            conn.close()

In [6]:
# Tools list
tools = [query_crm_database]

# Creates the object for the tool node.
tool_node = ToolNode(tools) 

In [7]:
# Define the function that creates a "runnable" agent from an LLM and a system prompt.
def create_runnable_agent(llm, system_prompt):
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            MessagesPlaceholder(variable_name = "messages"),
        ]
    )
    agent_runnable = prompt | llm.bind_tools(tools)
    return agent_runnable

In [11]:
# Defines the function of the Groq agent node responsible for interacting with the CRM.
def groq_agent_node(state: AgentState):
    print("\n *** Running the Groq Node (CRM) *** \n")
    try:
        # Initializes the LLM Groq with the model, temperature, and API key.
        llm_groq = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.2, groq_api_key=groq_api_key) 
        
        # Sets the system prompt with instructions for using the CRM tool
        system_prompt = """You are a CRM assistant named Groq (model Llama3).
            Your main function is to answer questions about clients and interactions by querying the CRM database.
            Use the 'query_crm_database' tool by providing a valid SQL SELECT query to retrieve the requested information.
            See the tool description for the database schema (tables: tb_clients, tb_interactions and their columns).
            Be direct and base your answers on the data returned by the tool. If the tool returns an error, inform the user.
            Do not invent information if it is not in the database.
        """
        
        agent_runnable = create_runnable_agent(llm_groq, system_prompt)
        print("Runnable Groq (CRM) created. Invoking...")
        response = agent_runnable.invoke({"messages": state['messages']})
        print(f"Node Groq (CRM) Retrieved Response: Type = {type(response)}, Content = '{response.content[:50]}...'")
        
        if hasattr(response, 'tool_calls') and response.tool_calls:
            print(f"Node Groq (CRM) is calling the tool: {response.tool_calls}")
        
        return {"messages": [response]}

    except Exception as e:
        print(f"!!! Groq Node Error (CRM): {e} !!!")
        print(f"An error occurred while conecting the Groq API: {e}")
        error_msg = AIMessage(content = f"[GROQ INTERNAL ERROR]: It was not possible to process with Groq. Detail: {e}", name = "ErrorGroq")
        return {"messages": [error_msg]}

In [12]:
# Defines the function of the OpenAI agent node responsible for interacting with the CRM.
def openai_agent_node(state: AgentState):
    print("\n--- Running the OpenAI Node (CRM) ---")
    try:
        llm_openai = ChatOpenAI(temperature=0.2, openai_api_key=openai_api_key, model_name="gpt-3.5-turbo")
        
        system_prompt = """You are an experienced CRM assistant called OpenAI (GPT model).
            Your goal is to assist the user with information from the CRM database.
            Use the 'query_crm_database' tool to execute SQL SELECT queries and retrieve data about customers or interactions.
            Refer to the tool's description to understand the database schema (tables: tb_clients, tb_interactions; relevant columns such as name, email, status, interaction_date, type, notes).
            Formulate precise SQL SELECT queries based on the user's question.
            Present the results clearly. If you encounter a tool error, report it.
            If the information is not available, indicate this clearly.
        """
        
        agent_runnable = create_runnable_agent(llm_openai, system_prompt)
        print("Runnable OpenAI (CRM) created. Invoking...")
        response = agent_runnable.invoke({"messages": state['messages']})
        print(f"Node OpenAI (CRM) Retrieved Response: Type={type(response)}, Content='{response.content[:50]}...'")

        if hasattr(response, 'tool_calls') and response.tool_calls:
            print(f"Node OpenAI (CRM) is calling the tool: {response.tool_calls}")

        return {"messages": [response]}

    except Exception as e:
        print(f"!!! OpenAI Node Error (CRM): {e} !!!")
        print(f"An error occurred while conecting the OpenAI API: {e}")
        error_msg = AIMessage(content=f"[GROQ INTERNAL ERROR]: It was not possible to process with OpenAI. Detail: {e}", name="ErrorOpenAI")
        return {"messages": [error_msg]}

In [10]:
# Function for the routing node
# The routing logic will be in the next function
# Even though it doesn't have processing logic, it acts as an explicit routing node, making it clear in the graph where the central decision occurs
# In agent flow graphs, it's good practice to have explicit nodes that act as hubs or routers, even if they don't modify the state
def route_junction_node(state: AgentState) -> dict:
    print("--- Routing Junction Node (No State Change) ---")
    return {}

In [13]:
# Define the function responsible for deciding where the router should send the next message
def router_logic(state: AgentState) -> str:
    print("\n--- Routing Logic Function (Deciding Next Step) ---")
    messages = state['messages']
    last_message = messages[-1] if messages else None

    if not last_message:
        print("Logical Decision: No messages in the state, ending.")
        return "__end__"

    print(f"Router analyzing last message: Type={type(last_message).__name__}, Content='{last_message.content[:80]}...'")

    if isinstance(last_message, AIMessage) and hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        print("Logical Decision: Last AI message has 'tool_calls'. Routing to Tools.")
        return "tools"

    if isinstance(last_message, AIMessage):
        print("Logical Decision: Final AI response received (without tool_calls). Ending the current loop.")
        return "__end__"

    if isinstance(last_message, HumanMessage):
        user_input_current = last_message.content.lower()
        print(f"Analyzing last human message for mentions: '{user_input_current}'")

        if "@openai" in user_input_current:
            print("Logical Decision: Routing to OpenAI (explicit mention in the last message)")
            return "openai_agent"

        elif "@groq" in user_input_current:
            print("Logical Decision: Routing to Groq (explicit mention in the last message)")
            return "groq_agent"

    if isinstance(last_message, ToolMessage):
        print("Logical Decision: Tool result received, routing to an agent (via alternation)...")

    ai_message_count = sum(1 for msg in messages if isinstance(msg, AIMessage))
    print(f"Current AI message count for alternation: {ai_message_count}")

    if ai_message_count % 2 == 0:
        print(f"Logical Decision: Routing to Groq (default/alternating)")
        return "groq_agent"

    else:
        print(f"Logical Decision: Routing to OpenAI (default/alternating)")
        return "openai_agent"

In [14]:
# Define the function responsible for compiling the agent's state and transition graph.
def compile_graph():
    workflow = StateGraph(AgentState)
    workflow.add_node("openai_agent", openai_agent_node)
    workflow.add_node("groq_agent", groq_agent_node)
    workflow.add_node("tools", tool_node)
    workflow.add_node("router", route_junction_node)

    # Connecting the START entry point to the routing node
    workflow.add_edge(START, "router")
    
    # Configuring conditional edges exiting the router based on routing logic.
    workflow.add_conditional_edges(
        "router",
        router_logic,
        {
            "tools": "tools",
            "groq_agent": "groq_agent",
            "openai_agent": "openai_agent",
            "__end__": END
        },
    )
    
    workflow.add_edge("openai_agent", "router")
    workflow.add_edge("groq_agent", "router")
    workflow.add_edge("tools", "router")

    # Compiling the workflow into an executable application.
    app = workflow.compile()

    print("Graph compiled successfully!")
    return app

In [15]:
if not os.path.exists(DB_FILE):
    print(f"Error: The database file '{DB_FILE}' was not found.")
    print("Please run the 'create_crm_db.py' script in the same directory to create the database, and then reload this page.")

print("Initializing the graph for the first time...")

try:
    app = compile_graph()
    
    # If the chat history does not exist in the session, it initializes with a welcome message.
    chat_history = [AIMessage(content="Hello! I'm your CRM assistant. Ask me about clients or interactions (e.g., 'Which clients are active?', 'Show oão Smith's interactions').")]
    
    print(chat_history)
    # Displays a success message after initializing the graph.
    print("CRM graph initialized.")

except Exception as e:
    # In case of a critical error in the graph construction, display an error message and the exception.
    print(f"Critical error when building the CRM graph.: {e}")
    print(e)

Initializing the graph for the first time...
Graph compiled successfully!
[AIMessage(content="Hello! I'm your CRM assistant. Ask me about clients or interactions (e.g., 'Which clients are active?', 'Show oão Smith's interactions').", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
CRM graph initialized.


In [16]:
for i, msg in enumerate(chat_history):
    role = "ai" if isinstance(msg, AIMessage) else ("tool" if isinstance(msg, ToolMessage) else "user")
    avatar_icon = "👤"
    sender_name = "User"
    message_role_for_streamlit = "user"

    if role == "ai":
        message_role_for_streamlit = "assistant"
        
        ai_message_index = sum(1 for m in chat_history[:i] if isinstance(m, AIMessage))
        is_groq_explicit = "@groq" in msg.content.lower()
        is_openai_explicit = "@openai" in msg.content.lower()
        
        msg_name = getattr(msg, 'name', None)
        
        # If it's a Groq or an alternating pattern with an even index and no name, set the Groq avatar.
        if is_groq_explicit or (not is_openai_explicit and ai_message_index % 2 == 0 and not msg_name):
                avatar_icon = "🦙"
                sender_name = "Groq (Llama3)"
        
        # If it's OpenAI or an odd-indexed, nameless toggle, set the OpenAI avatar.
        elif is_openai_explicit or (not is_groq_explicit and ai_message_index % 2 != 0 and not msg_name):
                avatar_icon = "🤔"
                sender_name = "OpenAI (GPT)"
        
        # If a custom name is included in the message, use the system avatar.
        elif msg_name:
                avatar_icon = "⚠️"
                sender_name = f"System ({msg_name})"
        
        # Otherwise, it uses a generic assistant avatar.
        else:
                avatar_icon = "🤖"
                sender_name = "Assistant"
    
    # If the message is from Tool, adjust role, avatar, and name.
    elif role == "tool":
            message_role_for_streamlit = "assistant"
            avatar_icon = "🛠️"
            sender_name = "Tool"


    if role == "tool":
        tool_name = getattr(msg, 'name', 'query_crm_database')
        print(f"**Tool result ({tool_name})**:")
        print(f"{msg.content}")
        print(f"Call ID: {msg.tool_call_id}")
        
    # If the message is from AI, it displays the content and calls to the Tool if available.
    elif role == "ai":
        print(f"**{sender_name}:**")
        if getattr(msg, 'tool_calls', None):
                print(f"*Calling Tool(s):*")
                print([{'name': tc.get('name', 'N/A'), 'args': tc.get('args', {})} for tc in msg.tool_calls])
        print(msg.content)
        
    # If it's a message from the User, it displays the text directly.
    else:
        print(msg.content)

**Groq (Llama3):**
Hello! I'm your CRM assistant. Ask me about clients or interactions (e.g., 'Which clients are active?', 'Show oão Smith's interactions').


In [17]:
# If the user enters a question in the chat
prompt = "Which clients are active?"

# Adds the human message to the chat history in the session
chat_history.append(HumanMessage(content=prompt))

In [18]:
# Memory Management and State Initialization for AI Agents
# If chat history exists and the last message is from the user
if chat_history and isinstance(chat_history[-1], HumanMessage):
    # Stores the last human message
    last_human_message = chat_history[-1]
    
    # Prepare the current state with all messages
    current_state = {"messages": chat_history}
    
    print("CRM quering and thinking...")
        
    final_state = None
    try:
        # Invokes the graph from the current state.
        final_state = app.invoke(current_state)
        
        # If the graph returned a valid state containing messages
        if final_state and "messages" in final_state:            
            new_messages = final_state["messages"][len(current_state["messages"]):]
            
            if new_messages:
                chat_history.extend(new_messages)
            else:
                print("The graph did not return any new messages this time.", icon="🤔")
        else:
            print("The graph returned an invalid state.", icon="error")
            chat_history.append(AIMessage(content="Sorry, an internal error occurred in the graph state."))
    
    except Exception as e:
        print(f"Error during graph execution.: {e}")
        chat_history.append(AIMessage(content=f"Sorry, an error occurred.: {e}"))
    

CRM quering and thinking...
--- Routing Junction Node (No State Change) ---

--- Routing Logic Function (Deciding Next Step) ---
Router analyzing last message: Type=HumanMessage, Content='Which clients are active?...'
Analyzing last human message for mentions: 'which clients are active?'
Current AI message count for alternation: 1
Logical Decision: Routing to OpenAI (default/alternating)

--- Running the OpenAI Node (CRM) ---
Runnable OpenAI (CRM) created. Invoking...
Node OpenAI (CRM) Retrieved Response: Type=<class 'langchain_core.messages.ai.AIMessage'>, Content='...'
Node OpenAI (CRM) is calling the tool: [{'name': 'query_crm_database', 'args': {'sql_query': "SELECT name, email FROM tb_clients WHERE status = 'Active';"}, 'id': 'call_JUiPZSDeKO5mrWpaB58spGyz', 'type': 'tool_call'}]
--- Routing Junction Node (No State Change) ---

--- Routing Logic Function (Deciding Next Step) ---
Router analyzing last message: Type=AIMessage, Content='...'
Logical Decision: Last AI message has 'too